# 01 · HMM / CRF 序列标注 —— 在“张三在北京”上学会“读句”

**家族位置**：`04_Sequence_Models` 第 1 站（传统序列）。02/03 学“看图”（空间同时看），本章学“读句”（时间先后看）——“我没去”vs“我去没”顺序一反意思全反，模型必须记住前后依赖。

**学习目标**
1. HMM 生成式 vs CRF 判别式：一个算“怎么生成这句话”，一个算“给定这句话怎么标最对”
2. 维特比解码：句长 T=8 时用动态规划从 7^8 条路径里挑最优，不用枚举
3. 同数据同标注对照：toy 中文 NER 上 HMM/CRF 谁记得住上下文，谁被标注偏置误导
4. 实体 F1 的意义：字符对了不算对，整段人名/地名边界对了才算对

## 1. 原理：从“单字分类”到“序列标注”

### 通俗理解

**一句话**：单字分类是“这字像人名吗”，序列标注是“整句一起看，前后要自洽——B-PER 后面只能跟 I-PER，不能跟 I-LOC”。

**比喻**：单字分类像让 7 个人各自猜；序列标注像让 7 个人手拉手猜——前一个人说“我是 B-PER（人名开头）”，后一个人就不敢说“我是 I-LOC（地名延续）”，否则整句不通。CRF 就是给手拉手加“转移分”，HMM 给“转移概率”。

### 结构账

```
单字分类：  x1→softmax  x2→softmax  ...  各管各的
HMM  生成：  π(起手)·A(转移)·B(发射)  联合概率 p(x,y)=p(y1)·∏p(yt|yt-1)·∏p(xt|yt)，频率估计+拉普拉斯
CRF  判别：  score(x,y)=∑emit[xt,yt]+∑trans[yt-1,yt]  条件概率 p(y|x)=exp(score)/Z(x)，配分函数 Z 前向算法，梯度靠前后向
维特比：   dp[t,s]=max_{prev}(dp[t-1,prev]+trans[prev,s])+emit[xt,s]  最优路径回溯
```

- **发射/emit**：字-标签亲和度（如“张”爱跟 B-PER）
- **转移/trans**：标签-标签亲和度（如 B-PER→I-PER 高分，B-PER→I-LOC 低分）
- **评估**：字符准确率 vs **实体 F1**（严格 span 匹配，边界错也算错）

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import RAW_SENTENCES, TAGS, TAG2ID, ID2TAG, get_splits, build_vocab, tag_distribution
from common.models import HMMSegmenter, LinearCRF
from common.engine import token_accuracy, entity_f1
from common.utils import set_seed, setup_chinese_font, draw_tokens, TAG_COLORS

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)

train_X, train_Y, test_X, test_Y = get_splits()
all_sents = train_X + test_X
vocab = build_vocab(all_sents)
print(f"train {len(train_X)} 句 / test {len(test_X)} 句 | 词表 {len(vocab)} 字符 | 标签 {TAGS}")
print("标签分布 train:", tag_distribution(train_Y))
print("首句:", train_X[0], "→", train_Y[0])


## 2. 数据：12 句 toy 中文 NER（字符级 BIO）

8 句训练 + 4 句测试，字符级标注，O 占多数符合真实分布。测试句是训练字符的新组合，考泛化而非背诵。

In [ ]:
# fig0：4 句标注可视化（2 训练 + 2 测试）
import matplotlib
samples = [
    (train_X[0], train_Y[0], "训练·张三在北京工作"),
    (train_X[4], train_Y[4], "训练·小明在清华大学读书"),
    (test_X[1], test_Y[1], "测试·王五来自杭州"),
    (test_X[3], test_Y[3], "测试·华为的陈明"),
]
fig, axes = plt.subplots(4, 1, figsize=(10, 5.2))
for ax, (toks, tags, title) in zip(axes, samples):
    draw_tokens(ax, toks, tags, title=title)
fig.suptitle("Toy NER 标注（BIO，B-开头 I-延续 O-其他）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

# 词表前瞻
print("词表大小:", len(vocab), "示例:", list(vocab.items())[:8])


## 3. HMM：频率估计 + 拉普拉斯平滑 + 对数 Viterbi

生成式，pi/A/B 频率估计，已验证在训练句上能复盘。

In [ ]:
def encode(X):
    return [[vocab.get(ch, 0) for ch in s] for s in X]

train_X_ids = encode(train_X)
test_X_ids = encode(test_X)
train_Y_ids = [[TAG2ID[t] for t in seq] for seq in train_Y]
test_Y_ids = [[TAG2ID[t] for t in seq] for seq in test_Y]

hmm = HMMSegmenter(TAGS, len(vocab), alpha=1.0).fit(train_X_ids, train_Y_ids)
print("HMM pi/A/B 已估计（对数）")
# fig1：发射热力（Top 字符×标签，选择性展示）
import matplotlib.colors as mcolors
# 挑 12 个高频字符
from collections import Counter
chars_all = [ch for s in all_sents for ch in s]
common_chars = [c for c,_ in Counter(chars_all).most_common(12)]
ids = [vocab[c] for c in common_chars]
B = np.exp(hmm.B)  # 转回概率作展示
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(B[:, ids].T, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(TAGS))); ax.set_xticklabels(TAGS, fontsize=8, rotation=20)
ax.set_yticks(range(len(common_chars))); ax.set_yticklabels(common_chars, fontsize=9)
ax.set_title("HMM 发射概率 p(字|标签)（Top12 字符）")
plt.colorbar(im, ax=ax, shrink=0.9, label="p")
plt.tight_layout()
plt.savefig(FIGS / "fig1_hmm_emit.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：转移矩阵热力
fig, ax = plt.subplots(figsize=(5, 4.2))
A = np.exp(hmm.A)
im = ax.imshow(A, cmap="Oranges", vmin=0)
ax.set_xticks(range(len(TAGS))); ax.set_xticklabels(TAGS, fontsize=7, rotation=25)
ax.set_yticks(range(len(TAGS))); ax.set_yticklabels(TAGS, fontsize=7)
ax.set_title("HMM 转移概率 p(后一标签|前一标签)")
for i in range(len(TAGS)):
    for j in range(len(TAGS)):
        ax.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=6, color="black" if A[i,j]<0.5 else "white")
plt.colorbar(im, ax=ax, shrink=0.9)
plt.tight_layout()
plt.savefig(FIGS / "fig2_hmm_trans.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. CRF：判别式条件似然训练（前向配分 + 维特比解码）

与 HMM 同数据同标签对照，SGD 80 轮。

In [ ]:
crf = LinearCRF(len(TAGS), len(vocab), seed=0)
crf.fit(train_X_ids, train_Y_ids, epochs=80, lr=0.12, verbose=True)
print("CRF 训练完成（emit/trans 已学）")

# fig3：CRF 转移权重热力（可为负，红正蓝负）
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(crf.trans, cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(len(TAGS))); ax.set_xticklabels(TAGS, fontsize=7, rotation=25)
ax.set_yticks(range(len(TAGS))); ax.set_yticklabels(TAGS, fontsize=7)
ax.set_title("CRF 转移权重 trans[prev, cur]（判别式，可正可负）")
for i in range(len(TAGS)):
    for j in range(len(TAGS)):
        ax.text(j, i, f"{crf.trans[i,j]:.1f}", ha="center", va="center", fontsize=6, color="black" if abs(crf.trans[i,j])<1.2 else "white")
plt.colorbar(im, ax=ax, shrink=0.9)
plt.tight_layout()
plt.savefig(FIGS / "fig3_crf_trans.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 主实验：同数据同标注，HMM vs CRF 谁读得更准

字符准确率 + 实体 F1（严格 span）双口径。

In [ ]:
def predict_all(model, X_ids):
    return [model.viterbi(x) for x in X_ids]

def ids2tags(ids_list):
    return [[ID2TAG[i] for i in seq] for seq in ids_list]

hmm_pred_ids = predict_all(hmm, test_X_ids)
crf_pred_ids = predict_all(crf, test_X_ids)
# 训练集也看，验证是否欠拟合
hmm_train_pred = predict_all(hmm, train_X_ids)
crf_train_pred = predict_all(crf, train_X_ids)

# 字符准确率
print("训练 字符acc：HMM", round(token_accuracy(ids2tags(hmm_train_pred), train_Y),4), "CRF", round(token_accuracy(ids2tags(crf_train_pred), train_Y),4))
print("测试 字符acc：HMM", round(token_accuracy(ids2tags(hmm_pred_ids), test_Y),4), "  CRF", round(token_accuracy(ids2tags(crf_pred_ids), test_Y),4))
# 实体 F1
for name, pred in [("HMM", ids2tags(hmm_pred_ids)), ("CRF", ids2tags(crf_pred_ids))]:
    prec, rec, f1 = entity_f1(pred, test_Y)
    print(f"{name} 实体 P/R/F1: {prec:.3f}/{rec:.3f}/{f1:.3f}")

# 逐句打印对照
for i, (toks, gold, hp, cp) in enumerate(zip(test_X, test_Y, ids2tags(hmm_pred_ids), ids2tags(crf_pred_ids))):
    print(f"\n句{i+1} {' '.join(toks)}")
    print(" gold", gold)
    print("  HMM", hp, "✓" if hp==gold else "✗")
    print("  CRF", cp, "✓" if cp==gold else "✗")


In [ ]:
# fig4：测试 F1/字符acc 柱状同台
import numpy as np
labels = ["字符acc", "实体F1"]
h_acc = token_accuracy(ids2tags(hmm_pred_ids), test_Y)
c_acc = token_accuracy(ids2tags(crf_pred_ids), test_Y)
_, _, h_f1 = entity_f1(ids2tags(hmm_pred_ids), test_Y)
_, _, c_f1 = entity_f1(ids2tags(crf_pred_ids), test_Y)
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2)
w = 0.34
b1 = ax.bar(x - w/2, [h_acc, h_f1], w, label="HMM", color="#4C72B0")
b2 = ax.bar(x + w/2, [c_acc, c_f1], w, label="CRF", color="#55A868")
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{b.get_height():.2f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.12); ax.set_ylabel("分数")
ax.set_title("HMM vs CRF（测试集：字符准确率 / 实体F1 同台）")
ax.legend(); plt.tight_layout()
plt.savefig(FIGS / "fig4_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# fig5：2 句测试对比条带（gold vs HMM vs CRF）
# 选第 1、4 句
pick_idx = [0, 3]
fig, axes = plt.subplots(len(pick_idx)*3, 1, figsize=(10, 5.6), sharex=False)
# axes 是 6 行：每句 gold/HMM/CRF 各一行
row = 0
for j in pick_idx:
    toks = test_X[j]
    for tags, name in [(test_Y[j], "Gold"), (ids2tags(hmm_pred_ids)[j], "HMM"), (ids2tags(crf_pred_ids)[j], "CRF")]:
        draw_tokens(axes[row], toks, tags, title=f"{name} · {''.join(toks)}", fontsize=10)
        row += 1
fig.suptitle("测试句逐字对照（Gold vs HMM vs CRF）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig5_preds.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. 总结与下一步

**本项目收获**

1. 生成式 HMM vs 判别式 CRF：前者算联合概率、频率估计；后者直接优化条件似然、转移可正可负更灵活
2. 维特比是序列标注的标配解码器，HMM/CRF 共用同一套动态规划
3. 实体 F1 比字符准确率更严——边界错也算错，这才反映 NER 真正难度
4. toy 上 CRF 拟合更强的原因：判别式不受“每标签发射分布必归一”的约束

**下一步**：`02_RNN_LSTM_GRU_TextCls`——把“统计”换成“神经网络的记忆”（RNN→LSTM/GRU），在长文本分类上看谁记得住。